In [8]:
import random

board = [[1, 2, 3], [4, 0, 6], [7, 5, 8]]
GOAL  = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]


def in_bang(board, title=""):
    if title:
        print(title)
    for row in board:
        dong = ""
        for x in row:
            if x == 0:
                dong = dong + "_ "
            else:
                dong = dong + str(x) + " "
        print(dong)
    print()

def tim_o_trong(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def di_chuyen(board, action):
    new_board = []
    for row in board:
        new_board.append(row[:])

    r, c = tim_o_trong(new_board)

    if action == 'U':
        nr, nc = r - 1, c
    elif action == 'D':
        nr, nc = r + 1, c
    elif action == 'L':
        nr, nc = r, c - 1
    else:
        nr, nc = r, c + 1

    temp = new_board[r][c]
    new_board[r][c] = new_board[nr][nc]
    new_board[nr][nc] = temp

    return new_board


agent_memory = {
    "state":  board,
    "model":  {"goal": GOAL},
    "rules":  [
        ("xong",    "stop"),
        ("default", "random"),
    ],
    "action": None,
    "lich_su": []
}


def update_state(state, action, percept, model):
    if action is None:
        ban_sao = []
        for row in percept:
            ban_sao.append(row[:])
        return ban_sao

    predicted = di_chuyen(state, action)

    if predicted == percept:
        return predicted

    ban_sao = []
    for row in percept:
        ban_sao.append(row[:])
    return ban_sao


def rule_match(state, rules, model):
    goal = model["goal"]

    da_dung = True
    for r in range(3):
        for c in range(3):
            if state[r][c] != goal[r][c]:
                da_dung = False
                break

    for condition, action in rules:
        if condition == "xong" and da_dung == True:
            return condition, action
        if condition == "default":
            return condition, action

    return "default", "random"


def cam_bien(board, lich_su):
    r, c = tim_o_trong(board)
    tat_ca = []

    if r > 0:
        tat_ca.append('U')
    if r < 2:
        tat_ca.append('D')
    if c > 0:
        tat_ca.append('L')
    if c < 2:
        tat_ca.append('R')

    hop_le = []
    for action in tat_ca:
        trang_thai_moi = di_chuyen(board, action)
        if trang_thai_moi not in lich_su:
            hop_le.append(action)

    return hop_le


def model_based_reflex_agent(percept):
    mem = agent_memory

    mem["state"] = update_state(mem["state"], mem["action"], percept, mem["model"])

    if mem["state"] not in mem["lich_su"]:
        mem["lich_su"].append(mem["state"])

    condition, action = rule_match(mem["state"], mem["rules"], mem["model"])

    if action == "stop":
        return None, condition

    if action == "random":
        hop_le = cam_bien(mem["state"], mem["lich_su"])

        if len(hop_le) == 0:
            mem["lich_su"] = []
            hop_le = cam_bien(mem["state"], mem["lich_su"])

        action = random.choice(hop_le)

    mem["action"] = action
    return action, condition


if __name__ == '__main__':
    current_board = []
    for row in board:
        current_board.append(row[:])

    in_bang(current_board, "Trạng thái ban đầu:")
    in_bang(GOAL, "Trạng thái đích:")

    for step in range(30):
        action, rule = model_based_reflex_agent(current_board)

        if action is None:
            print("Bước " + str(step + 1) + ": [HOÀN THÀNH]")
            break

        current_board = di_chuyen(current_board, action)
        print("Bước " + str(step + 1) + ": rule='" + rule + "' → action='" + action)
        in_bang(current_board)

Trạng thái ban đầu:
1 2 3 
4 _ 6 
7 5 8 

Trạng thái đích:
1 2 3 
4 5 6 
7 8 _ 

Bước 1: rule='default' → action='D
1 2 3 
4 5 6 
7 _ 8 

Bước 2: rule='default' → action='R
1 2 3 
4 5 6 
7 8 _ 

Bước 3: [HOÀN THÀNH]
